In [ ]:
import pandas as pd
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "app").exists():
            return candidate
    raise FileNotFoundError("Could not locate the BTS Song Atlas project root.")


PROJECT_ROOT = find_project_root()


In [ ]:
BASE_DIR = PROJECT_ROOT
PROCESSED_DIR = BASE_DIR / "data" / "processed"

albums = pd.read_csv(PROCESSED_DIR / "spotify_albums_clean.csv")
tracks = pd.read_csv(PROCESSED_DIR / "spotify_tracks_clean.csv")
lyrics = pd.read_csv(PROCESSED_DIR / "song_lyrics_clean.csv")

In [ ]:
#merge tracks + albums
song_master = tracks.merge(
    albums[["album_id", "release_date", "release_year", "total_tracks", "image_url"]],
    on="album_id",
    how="left"
)

In [ ]:
# merge lyrics
song_master = song_master.merge(
    lyrics[[
        "track_id",
        "lyrics_clean", 
        "character_count",
        "word_count", 
        "genius_title",
        "genius_artist",
        "genius_url"
    ]],
    on="track_id",
    how="left"
)

In [ ]:
print("Song Master Shape:", song_master.shape)

print("\n" + "---" * 30)

print("Missing Values")
print(song_master.isna().sum().sort_values(ascending=False))

print("\n" + "---" * 30)

lyrics_count = song_master["lyrics_clean"].notna().sum()

print(f"Tracks with lyrics: {lyrics_count}")
print(f"Tracks without lyrics: {len(song_master) - lyrics_count}")
print(f"Coverage: {lyrics_count / len(song_master):.1%}")

In [ ]:
# Duplicate track IDs

print("Unique track IDs:", song_master["track_id"].is_unique)

In [ ]:
# Duplicate rows

print("Duplicate rows:", song_master.duplicated().sum())

In [ ]:
# Missing album metadata

print(
    "Tracks missing release year:",
    song_master["release_year"].isna().sum()
)

In [ ]:
song_master.to_csv(
    PROCESSED_DIR / "song_master.csv",
    index=False
)

print("Saved:", PROCESSED_DIR / "song_master.csv")
print("Shape:", song_master.shape)

In [ ]:
check = pd.read_csv(PROCESSED_DIR / "song_master.csv")

print(check.shape)
check.head()